Install Dependencies

In [ ]:
!pip install transformers datasets sentencepiece peft scikit-learn -q


Load data

In [ ]:
from datasets import load_dataset
print("Loading dataset...")
dataset = load_dataset("lightonai/SwissProt-EC-leaf")
print(dataset)
print("\nFirst example:")
print(dataset["train"][0])

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/910 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/72.0M [00:00<?, ?B/s]

...

data/dev-00000-of-00001.parquet:   0%|          | 0.00/9.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/178302 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22183 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/23010 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['seq', 'labels', 'labels_str', 'id'],
        num_rows: 178302
    })
    test: Dataset({
        features: ['seq', 'labels', 'labels_str', 'id'],
        num_rows: 22183
    })
    dev: Dataset({
        features: ['seq', 'labels', 'labels_str', 'id'],
        num_rows: 23010
    })
})

First example:
{'seq': 'MKFSEQWLRGWVSPQVDRDALVARLSMAGLEVDSVTPAAGVFSGVVVGEVLSTEQHPDADKLRVCQVSNGAETFQVVCGAPNVRPGLKIPFAMIGAELPGDFKIKKAKLRGVESNGMLCSQAELQIGEGNDGLMELPADASVGEDFRVYLDLEDASIEVDLTPNRGDCLSLAGLAREVGALYDAPVTRPVVMAVPAAHDEVRSVEVLAPAACPRYLGRVIRNVDLSRPTPLWMVERLRRAEVRSIDAAVDITNYVMLELGQPLHAFDLAEINGGIRVRMAEEGEKLVLLDGQEVSLRSDTLVVADHTRALAIAGVMGGEHSGVSATTRDVFLESAFFDQIAVAGKARSYGLHTDASHRYERGVDWQLAREAMERATGLLLEITGGEAGPIIETVSEQHLPSIAPITLRAQRITQMLGMEMDSAEVERLLNALGLKVSADGAGQWRVEVPSHRFDISLEVDLIEELARLYGYNRLPVRYPQARLAPQAKAEARSDLPELRRLLVARGYQEAITYSFIDPKQFELFNPGVEPLLLANPISNDMAAMRSSLWPGLVKALQHNLNRQQDRVRLFESGLRFVGQLEGLKQEPMIAGVVCGSRLPEGWAQGRDTVDFFDVKADVEAVLGFAGALDQFTF

In [ ]:
# Create manageable subsets for fast experimentation

train_data = dataset["train"].shuffle(seed=42).select(range(5000))
val_data = dataset["dev"].shuffle(seed=42).select(range(1000))
test_data = dataset["test"].shuffle(seed=42).select(range(1000))

print(train_data)
print(val_data)
print(test_data)

Dataset({
    features: ['seq', 'labels', 'labels_str', 'id'],
    num_rows: 5000
})
Dataset({
    features: ['seq', 'labels', 'labels_str', 'id'],
    num_rows: 1000
})
Dataset({
    features: ['seq', 'labels', 'labels_str', 'id'],
    num_rows: 1000
})


In [ ]:
import time
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# -----------------------------
# 1. Load real SwissProt dataset
# -----------------------------
dataset = load_dataset("lightonai/SwissProt-EC-leaf")

train_data = dataset["train"].shuffle(seed=42).select(range(5000))
val_data = dataset["dev"].shuffle(seed=42).select(range(1000))
test_data = dataset["test"].shuffle(seed=42).select(range(1000))

# -----------------------------
# 2. Convert multi-label EC task into single-label task
# -----------------------------
def first_label(example):
    example["label"] = example["labels"][0]
    return example

train_data = train_data.map(first_label)
val_data = val_data.map(first_label)
test_data = test_data.map(first_label)

# Build label map from training labels only
train_labels = sorted(set(train_data["label"]))
label2id = {label: i for i, label in enumerate(train_labels)}

# Keep only validation/test labels seen during training
val_data = val_data.filter(lambda x: x["label"] in label2id)
test_data = test_data.filter(lambda x: x["label"] in label2id)

def remap_label(example):
    example["label"] = label2id[example["label"]]
    return example

train_data = train_data.map(remap_label)
val_data = val_data.map(remap_label)
test_data = test_data.map(remap_label)

num_labels = len(label2id)

print("Number of classes:", num_labels)
print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))

# -----------------------------
# 3. Tokenize protein sequences
# -----------------------------
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["seq"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_tok = train_data.map(tokenize, batched=True)
val_tok = val_data.map(tokenize, batched=True)
test_tok = test_data.map(tokenize, batched=True)

# Keep only what Trainer needs
remove_cols = ["seq", "labels", "labels_str", "id"]
train_tok = train_tok.remove_columns(remove_cols)
val_tok = val_tok.remove_columns(remove_cols)
test_tok = test_tok.remove_columns(remove_cols)

train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

# -----------------------------
# 4. Metrics
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

# -----------------------------
# 5. Train linear probe
# -----------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

# Freeze all ESM2 layers except classifier
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

args = TrainingArguments(
    output_dir="./esm2_linear_probe",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics
)

start = time.time()
trainer.train()
runtime = time.time() - start

test_results = trainer.evaluate(test_tok)

print("\n==============================")
print("ESM2 LINEAR PROBE RESULTS")
print("==============================")
print(test_results)
print("Runtime seconds:", runtime)

Number of classes: 1142
Train: 5000
Val: 901
Test: 872


model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,6.260696,6.140940,0.066593,0.012448
2,5.635526,5.458884,0.106548,0.026164


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['esm.encoder.layer.0.attention.LayerNorm.weight', 'esm.encoder.layer.0.attention.LayerNorm.bias', 'esm.encoder.layer.0.LayerNorm.weight', 'esm.encoder.layer.0.LayerNorm.bias', 'esm.encoder.layer.1.attention.LayerNorm.weight', 'esm.encoder.layer.1.attention.LayerNorm.bias', 'esm.encoder.layer.1.LayerNorm.weight', 'esm.encoder.layer.1.LayerNorm.bias', 'esm.encoder.layer.2.attention.LayerNorm.weight', 'esm.encoder.layer.2.attention.LayerNorm.bias', 'esm.encoder.layer.2.LayerNorm.weight', 'esm.encoder.layer.2.LayerNorm.bias', 'esm.encoder.layer.3.attention.LayerNorm.weight', 'esm.encoder.layer.3.attention.LayerNorm.bias', 'esm.encoder.layer.3.LayerNorm.weight', 'esm.encoder.layer.3.LayerNorm.bias', 'esm.encoder.layer.4.attention.LayerNorm.weight', 'esm.encoder.layer.4.attention.LayerNorm.bias', 'esm.encoder.layer.4.LayerNorm.weight', 'esm.encoder.layer.4.LayerNorm.bias', 'esm.encoder.layer.5.attention.LayerNorm.weight', 'esm.encoder.


ESM2 LINEAR PROBE RESULTS
{'eval_loss': 5.53911828994751, 'eval_accuracy': 0.08830275229357798, 'eval_macro_f1': 0.024086490741448374, 'eval_runtime': 2.043, 'eval_samples_per_second': 426.821, 'eval_steps_per_second': 26.921, 'epoch': 2.0}
Runtime seconds: 34.239609479904175


In [ ]:
import time
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

# FULL FINE-TUNING: do NOT freeze layers

args = TrainingArguments(
    output_dir="./esm2_full_finetune",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics
)

start = time.time()
trainer.train()
runtime = time.time() - start

test_results = trainer.evaluate(test_tok)

print("\n==============================")
print("ESM2 FULL FINE-TUNE RESULTS")
print("==============================")
print(test_results)
print("Runtime seconds:", runtime)

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,6.912150,6.869210,0.077691,0.028495
2,6.825419,6.807829,0.083241,0.033156


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['esm.encoder.layer.0.attention.LayerNorm.weight', 'esm.encoder.layer.0.attention.LayerNorm.bias', 'esm.encoder.layer.0.LayerNorm.weight', 'esm.encoder.layer.0.LayerNorm.bias', 'esm.encoder.layer.1.attention.LayerNorm.weight', 'esm.encoder.layer.1.attention.LayerNorm.bias', 'esm.encoder.layer.1.LayerNorm.weight', 'esm.encoder.layer.1.LayerNorm.bias', 'esm.encoder.layer.2.attention.LayerNorm.weight', 'esm.encoder.layer.2.attention.LayerNorm.bias', 'esm.encoder.layer.2.LayerNorm.weight', 'esm.encoder.layer.2.LayerNorm.bias', 'esm.encoder.layer.3.attention.LayerNorm.weight', 'esm.encoder.layer.3.attention.LayerNorm.bias', 'esm.encoder.layer.3.LayerNorm.weight', 'esm.encoder.layer.3.LayerNorm.bias', 'esm.encoder.layer.4.attention.LayerNorm.weight', 'esm.encoder.layer.4.attention.LayerNorm.bias', 'esm.encoder.layer.4.LayerNorm.weight', 'esm.encoder.layer.4.LayerNorm.bias', 'esm.encoder.layer.5.attention.LayerNorm.weight', 'esm.encoder.


ESM2 FULL FINE-TUNE RESULTS
{'eval_loss': 6.816403865814209, 'eval_accuracy': 0.0676605504587156, 'eval_macro_f1': 0.0325520330265447, 'eval_runtime': 2.3778, 'eval_samples_per_second': 366.72, 'eval_steps_per_second': 23.13, 'epoch': 2.0}
Runtime seconds: 87.06932950019836


In [ ]:
## FINAL BENCHMARK: Top-20 EC Classes

from collections import Counter
from datasets import load_dataset
# Reload clean data
dataset = load_dataset("lightonai/SwissProt-EC-leaf")

# Take first EC label only
def first_label(example):
    example["label"] = example["labels"][0]
    return example

train_all = dataset["train"].map(first_label)
val_all = dataset["dev"].map(first_label)
test_all = dataset["test"].map(first_label)

# Find top 20 most common labels in train
counts = Counter(train_all["label"])
top20 = [label for label, count in counts.most_common(20)]
print("Top 20 labels:", top20)

# Keep only top 20 labels
train_data = train_all.filter(lambda x: x["label"] in top20)
val_data = val_all.filter(lambda x: x["label"] in top20)
test_data = test_all.filter(lambda x: x["label"] in top20)

# Limit size so it runs fast
train_data = train_data.shuffle(seed=42).select(range(min(5000, len(train_data))))
val_data = val_data.shuffle(seed=42).select(range(min(1000, len(val_data))))
test_data = test_data.shuffle(seed=42).select(range(min(1000, len(test_data))))

# Remap labels to 0-19
label2id = {label: i for i, label in enumerate(top20)}

def remap(example):
    example["label"] = label2id[example["label"]]
    return example

train_data = train_data.map(remap)
val_data = val_data.map(remap)
test_data = test_data.map(remap)

num_labels = 20

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))
print("Num labels:", num_labels)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/910 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/72.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/8.88M [00:00<?, ?B/s]

...

Filter:   0%|          | 0/22183 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train: 5000
Val: 1000
Test: 1000
Num labels: 20


In [ ]:
import time
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["seq"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_tok = train_data.map(tokenize, batched=True)
val_tok = val_data.map(tokenize, batched=True)
test_tok = test_data.map(tokenize, batched=True)

remove_cols = ["seq", "labels", "labels_str", "id"]
train_tok = train_tok.remove_columns(remove_cols)
val_tok = val_tok.remove_columns(remove_cols)
test_tok = test_tok.remove_columns(remove_cols)

train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=20
)

# LINEAR PROBE: freeze ESM2; train only classifier
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

args = TrainingArguments(
    output_dir="./esm2_top20_linear_probe",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics
)

start = time.time()
trainer.train()
linear_runtime = time.time() - start

linear_test_results = trainer.evaluate(test_tok)

print("\n==============================")
print("ESM2 TOP-20 LINEAR PROBE RESULTS")
print("==============================")
print(linear_test_results)
print("Runtime seconds:", linear_runtime)

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/2209 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,2.111623,1.597400,0.543986,0.421413
2,1.296717,1.305554,0.657092,0.600781


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['esm.encoder.layer.0.attention.LayerNorm.weight', 'esm.encoder.layer.0.attention.LayerNorm.bias', 'esm.encoder.layer.0.LayerNorm.weight', 'esm.encoder.layer.0.LayerNorm.bias', 'esm.encoder.layer.1.attention.LayerNorm.weight', 'esm.encoder.layer.1.attention.LayerNorm.bias', 'esm.encoder.layer.1.LayerNorm.weight', 'esm.encoder.layer.1.LayerNorm.bias', 'esm.encoder.layer.2.attention.LayerNorm.weight', 'esm.encoder.layer.2.attention.LayerNorm.bias', 'esm.encoder.layer.2.LayerNorm.weight', 'esm.encoder.layer.2.LayerNorm.bias', 'esm.encoder.layer.3.attention.LayerNorm.weight', 'esm.encoder.layer.3.attention.LayerNorm.bias', 'esm.encoder.layer.3.LayerNorm.weight', 'esm.encoder.layer.3.LayerNorm.bias', 'esm.encoder.layer.4.attention.LayerNorm.weight', 'esm.encoder.layer.4.attention.LayerNorm.bias', 'esm.encoder.layer.4.LayerNorm.weight', 'esm.encoder.layer.4.LayerNorm.bias', 'esm.encoder.layer.5.attention.LayerNorm.weight', 'esm.encoder.


ESM2 TOP-20 LINEAR PROBE RESULTS
{'eval_loss': 1.3280324935913086, 'eval_accuracy': 0.6428571428571429, 'eval_macro_f1': 0.558385920470796, 'eval_runtime': 1.255, 'eval_samples_per_second': 435.064, 'eval_steps_per_second': 27.889, 'epoch': 2.0}
Runtime seconds: 14.887039184570312


In [ ]:
import time
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=20
)

# FULL FINE-TUNING: do not freeze any parameters

args = TrainingArguments(
    output_dir="./esm2_top20_full_finetune",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics
)

start = time.time()
trainer.train()
full_runtime = time.time() - start

full_test_results = trainer.evaluate(test_tok)

print("\n==============================")
print("ESM2 TOP-20 FULL FINE-TUNE RESULTS")
print("==============================")
print(full_test_results)
print("Runtime seconds:", full_runtime)

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,2.752558,2.570324,0.581688,0.508259
2,2.446028,2.438630,0.644524,0.599018


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['esm.encoder.layer.0.attention.LayerNorm.weight', 'esm.encoder.layer.0.attention.LayerNorm.bias', 'esm.encoder.layer.0.LayerNorm.weight', 'esm.encoder.layer.0.LayerNorm.bias', 'esm.encoder.layer.1.attention.LayerNorm.weight', 'esm.encoder.layer.1.attention.LayerNorm.bias', 'esm.encoder.layer.1.LayerNorm.weight', 'esm.encoder.layer.1.LayerNorm.bias', 'esm.encoder.layer.2.attention.LayerNorm.weight', 'esm.encoder.layer.2.attention.LayerNorm.bias', 'esm.encoder.layer.2.LayerNorm.weight', 'esm.encoder.layer.2.LayerNorm.bias', 'esm.encoder.layer.3.attention.LayerNorm.weight', 'esm.encoder.layer.3.attention.LayerNorm.bias', 'esm.encoder.layer.3.LayerNorm.weight', 'esm.encoder.layer.3.LayerNorm.bias', 'esm.encoder.layer.4.attention.LayerNorm.weight', 'esm.encoder.layer.4.attention.LayerNorm.bias', 'esm.encoder.layer.4.LayerNorm.weight', 'esm.encoder.layer.4.LayerNorm.bias', 'esm.encoder.layer.5.attention.LayerNorm.weight', 'esm.encoder.


ESM2 TOP-20 FULL FINE-TUNE RESULTS
{'eval_loss': 2.449094772338867, 'eval_accuracy': 0.6373626373626373, 'eval_macro_f1': 0.5723028766742131, 'eval_runtime': 1.3735, 'eval_samples_per_second': 397.511, 'eval_steps_per_second': 25.481, 'epoch': 2.0}
Runtime seconds: 39.21109437942505


In [ ]:
from collections import Counter
from datasets import load_dataset

dataset = load_dataset("lightonai/SwissProt-EC-leaf")

# Convert multilabel -> single label
def first_label(example):
    example["label"] = example["labels"][0]
    return example

train_all = dataset["train"].map(first_label)
val_all = dataset["dev"].map(first_label)
test_all = dataset["test"].map(first_label)

# Top 20 EC classes
counts = Counter(train_all["label"])
top20 = [label for label, count in counts.most_common(20)]

train_data = train_all.filter(lambda x: x["label"] in top20)
val_data = val_all.filter(lambda x: x["label"] in top20)
test_data = test_all.filter(lambda x: x["label"] in top20)

# 10% train
train_size = int(len(train_data) * 0.10)

# 20% val/test
val_size = int(len(val_data) * 0.20)
test_size = int(len(test_data) * 0.20)

train_data = train_data.shuffle(seed=42).select(range(train_size))
val_data = val_data.shuffle(seed=42).select(range(val_size))
test_data = test_data.shuffle(seed=42).select(range(test_size))

# Remap labels to 0-19
label2id = {label: i for i, label in enumerate(top20)}

def remap(example):
    example["label"] = label2id[example["label"]]
    return example

train_data = train_data.map(remap)
val_data = val_data.map(remap)
test_data = test_data.map(remap)

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/910 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/72.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/8.88M [00:00<?, ?B/s]

...

Filter:   0%|          | 0/22183 [00:00<?, ? examples/s]

Map:   0%|          | 0/2209 [00:00<?, ? examples/s]

Map:   0%|          | 0/557 [00:00<?, ? examples/s]

Map:   0%|          | 0/546 [00:00<?, ? examples/s]

Train: 2209
Val: 557
Test: 546


In [ ]:
# Reset clean subsets
train_data = dataset["train"].shuffle(seed=42).select(range(5000))
val_data = dataset["dev"].shuffle(seed=42).select(range(1000))
test_data = dataset["test"].shuffle(seed=42).select(range(1000))

def first_label(example):
    example["label"] = example["labels"][0]
    return example

train_data = train_data.map(first_label)
val_data = val_data.map(first_label)
test_data = test_data.map(first_label)

# Build label map from training labels only
train_labels = sorted(set(train_data["label"]))
label2id = {label: i for i, label in enumerate(train_labels)}

# Filter validation/test to labels seen during training
val_data = val_data.filter(lambda x: x["label"] in label2id)
test_data = test_data.filter(lambda x: x["label"] in label2id)

def remap_label(example):
    example["label"] = label2id[example["label"]]
    return example

train_data = train_data.map(remap_label)
val_data = val_data.map(remap_label)
test_data = test_data.map(remap_label)

num_labels = len(label2id)

print("Number of classes:", num_labels)
print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))
print(train_data[0])

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/901 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Number of classes: 1142
Train: 5000
Val: 901
Test: 872
{'seq': 'MGDTALHLVAAPALEATGLQVARGGRPLFRGLGFRLARGGLLCVRGANGSGKTTLLRTLAGLSRPHRGRILRAGRCIRQAANDHRGTLQYRGHRDGLRGALTVLENLQWQAALYHHRPDREAVRTALAGMGMARHLDTLASQLSQGQRRRVVLASLALFPRCRLWLLDEPQAALDVEGAERFHALLARHCQEGGAVVACSHQHLRIGGIPCDELWLSDPAPAGARSAGDRVTGTEA', 'labels': [2509], 'labels_str': "['EC:7.6.2.5']", 'id': 'Q0A808', 'label': 591}


In [ ]:
num_labels = len(set(train_data["label"]))

Train/val/test split

In [ ]:
import numpy as np

# The dataset already has splits
train_data = dataset["train"]
val_data = dataset["dev"]
test_data = dataset["test"]

print(f"Train: {len(train_data)} sequences")
print(f"Val:   {len(val_data)} sequences")
print(f"Test:  {len(test_data)} sequences")
print(f"EC classes: 4793")

split_info = {
    "train_size": len(train_data),
    "val_size": len(val_data),
    "test_size": len(test_data),
    "num_classes": 4793,
    "dataset": "lightonai/SwissProt-EC-leaf"
}

import json
with open("split_info.json", "w") as f:
    json.dump(split_info, f)


Train: 178302 sequences
Val:   23010 sequences
Test:  22183 sequences
EC classes: 4793
